In [1]:
import sys
from pathlib import Path
root = Path().resolve().parent.parent   
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from utils.grade_submit import submit_grade
# Initialize Otter
import otter
grader = otter.Notebook("lab02.ipynb")

<img src="data8logo.png" alt="Data 8 Logo" style="width: 15%; float: right; padding: 1%; margin-right: 2%;"/>

# Лабораторна робота 2: Операції з таблицями

Ласкаво просимо до Лабораторії 2!  Цього тижня ми навчимося імпортувати модуль і потренуємося працювати з таблицями. [Довідник Python](https://www.data8.org/fa26/reference/) містить інформацію, яка буде корисною для цієї лабораторії.

**Recommended Reading**:
 * [Introduction to Tables](https://www.inferentialthinking.com/chapters/03/4/Introduction_to_Tables)

Почнемо з налаштування тестів та імпорту, запустивши клітинку нижче.

In [6]:
# Не змінюйте цю клітинку; просто запустіть її.
import numpy as np
from datascience import *

# Не потрібно турбуватися про те, що означає цей код
from IPython.display import Javascript, display
display(Javascript(r"""
(() => {
  function pathLooksLikeTyping(e) {
    const path = e.composedPath ? e.composedPath() : [];
    for (const n of path) {
      if (!n) continue;
      if (n.tagName === 'INPUT' || n.tagName === 'TEXTAREA') return true;
      if (n.isContentEditable) return true;
      const role = n.getAttribute?.('role');
      if (role === 'textbox' || role === 'combobox' || role === 'searchbox') return true;
      const ariaMulti = n.getAttribute?.('aria-multiline');
      if (ariaMulti === 'true') return true;
      const cls = (n.className || "").toString().toLowerCase();
    }
    return false;
  }
  function handler(e) {
    if (e.key !== 'o' && e.key !== 'O') return;
    if (pathLooksLikeTyping(e)) {
      e.stopPropagation();
      if (e.stopImmediatePropagation) e.stopImmediatePropagation();
      if (e.nativeEvent?.stopImmediatePropagation) e.nativeEvent.stopImmediatePropagation();
      return;
    }
    e.preventDefault();
    e.stopPropagation();
    if (e.stopImmediatePropagation) e.stopImmediatePropagation();
    if (e.nativeEvent?.stopImmediatePropagation) e.nativeEvent.stopImmediatePropagation();
  }
  window.addEventListener('keydown', handler, true);
  window.addEventListener('keypress', handler, true);
  console.log("Installed: 'o' won't toggle output; 'o' should type inside any textbox-like UI (including JupyTutor).");
})();
"""))

<IPython.core.display.Javascript object>

# 1. Огляд: Основні елементи коду Python

Двома основними елементами коду Python є *вирази* та *інструкції* (*expressions* and *statements*). 

**Вираз** (**Expression**) — якщо порівнювати з мовою людини, вираз це як окрема фраза або частина речення, яка має конкретне значення. Приклад, вираз «смачне яблуко» (він має значення, але це не завершена думка). В Python вираз — це фрагмент коду, який

* є самостійним, тобто його доцільно писати в окремому рядку, і
* зазвичай обчислюється до певного значення.

Ось два вирази, які обидва дають у результаті 3:

    3
    5 - 2

Одним із важливих типів виразів є **вираз виклику** (**call expression**). Вираз виклику починається з **імені функції**, за яким у дужках йдуть **аргумент(и) цієї функції**. Функція повертає певне значення, залежно від своїх аргументів. Нижче наведено перелік деяких важливих математичних функцій.

| Функція | Опис                                                   |
|----------|---------------------------------------------------------------|
| `abs`      | Повертає абсолютне значення свого аргументу                    |
| `max`      | Повертає максимальне значення серед усіх своїх аргументів                      |
| `min`      | Повертає мінімальне значення серед усіх своїх аргументів                      |
| `pow`      | Підносить перший аргумент до степеня, заданого другим аргументом |
| `round`    | Округлює аргумент до найближчого цілого числа                     |

Ось два вирази виклику, які обидва дають результат 3:

    abs(2 - 5)
    max(round(2.8), min(pow(2, 10), -1 * pow(2, 10)))

Вираз `2 - 5` та два наведені вище вирази-виклики є прикладами **складних виразів**, тобто вони фактично є комбінаціями кількох простіших виразів.  `2 - 5` поєднує вирази `2` та `5` за допомогою віднімання.  У цьому випадку `2` та `5` називаються **підвиразами**, оскільки вони є виразами, що входять до складу більшого виразу.

**Statement (Інструкція)** —  це як повне речення (приклад речення-наказ: «З'їж смачне яблуко.» - це завершена дія), яке завершує певну думку і змушує комп'ютер виконати якусь дію (наприклад, створити змінну, запустити цикл або вивести текст на екран). Це повна команда для комп'ютера, яка щось виконує (наприклад, створює змінну, запускає цикл або перериває програму).  Наведені вище вирази є прикладами.

Інші інструкції *здійснюють якусь дію*, а не *мають значення*. Наприклад, **інструкція присвоєння** присвоює значення імені. 
Про це зручно думати так: ми **обчислюємо праву частину** знака рівності та **присвоюємо її лівій частині**. Ось кілька прикладів інструкцій присвоєння:

    height = 1.3
    the_number_five = abs(-5)
    absolute_height_difference = abs(height - 1.688)

Важливою ідеєю в програмуванні є те, що великі, цікаві речі можна побудувати, поєднуючи багато простих, нецікавих речей.  Ключ до розуміння складного фрагмента коду полягає в тому, щоб розкласти його на прості складові.

Наприклад, в останньому операторі вище відбувається багато чого, але насправді це лише поєднання кількох речей.  Ця ілюстрація описує, що саме відбувається.

<img src="statement.png">

**Завдання 1.1.** У наступній комірці присвойте ім’я `new_year` **більшому числу** з-поміж наступних двох чисел:

* **абсолютне значення** виразу $2^{6}-2^{11}-2^{5} - 7$, та
* $5 \times 13 \times 31 + 11$.

Спробуйте використати лише один рядок коду. Обов’язково перевірте свою роботу, виконавши тестову комірку після цього.


In [ ]:
new_year = ...
new_year

In [ ]:
grader.check("q11")

У запитанні вище ми попросили вас використати лише один рядок коду, оскільки воно стосується виключно математичних операцій. Однак більш складні завдання з програмування вимагатимуть більшої кількості кроків. Не завжди доцільно вміщувати всі ці кроки в один рядок, оскільки це може ускладнити читання коду та його налагодження (debug).

Належна практика програмування передбачає розбиття коду на менші кроки та використання відповідних імен. У подальшій частині цього курсу ви отримаєте чимало практики!

# 1.2 Практика програмування в Data 8: перейменування змінних

Зауважте, що важливо не використовувати імена змінних, які збігаються з іменами наявних функцій. Візьмемо, наприклад, функцію `max`.

Зараз ця функція працює, як очікується:

In [ ]:
print(max) #  just shows us that this is currently a function
max(5, 6, 7) #  returns 7, as expected

Якщо ми використовуємо `max` як назву змінної, ми більше не зможемо використовувати функцію max (тому не робіть цього!)

In [ ]:
max = 8 #  ти не повинен цього робити!
print(max) #  тепер це друкує число 8, оскільки це число було те, що було збережено в змінній max
max(5, 6, 7) #  це більше не працює!

Запустіть цю клітинку нижче, щоб повернути функцію max до нормального для подальшого використання:

In [ ]:
del max

команда del видаляє змінну max, і тепер max знову є функцією

# 2. Імпорт коду

Більшість завдань у програмуванні повторюють або схожі з тим, що робили раніше. Оскільки написання коду займає багато часу, варто, коли це можливо, використовувати опублікований код інших авторів. Замість копіювання та вставлення Python дозволяє нам **імпортувати модулі**. Модуль — це файл із кодом Python, у якому визначено змінні та функції. Імпортуючи модуль, ми отримуємо можливість використовувати його код у власному ноутбуці.

Python містить багато корисних модулів, які можна завантажити за допомогою однієї команди `import`.  Як перший приклад розглянемо модуль `math`. Модуль `math` надзвичайно корисний для обчислення математичних виразів у Python.

Припустимо, ми хочемо дуже точно обчислити площу кола з радіусом 5 метрів.  Для цього нам потрібна константа $\pi$, яка дорівнює приблизно 3,14.  На щастя, у модулі `math` вже визначено змінну `pi` (і нам не треба це робити власноруч якщо ми імпортуємо цю змінну з модуля `math`). Виконайте наступну комірку, щоб імпортувати модуль `math`:

In [ ]:
import math
radius = 5
area_of_circle = radius**2 * math.pi
area_of_circle

У наведеному вище коді рядок `import math` імпортує модуль `math`. Тепер ми можемо отримати доступ до будь-яких змінних або функцій, визначених у модулі `math`, ввівши `math`, потім крапку, а потім — ім’я потрібної змінної або функції.

    <module name>.<name>

**Завдання 2.1.** Модуль `math` також надає змінну `e` в якій збережена основа натурального логарифма, яка дорівнює приблизно 2,71. Обчисліть $e^{\pi}-\pi$ і надайте результату ім’я `near_twenty`.

*Пам’ятайте: ви також можете отримати доступ до `pi` з модуля `math`!*


In [ ]:
near_twenty = ...
near_twenty

In [ ]:
grader.check("q21")

## 2.2. Виклик функцій

У наведеному вище питанні ви зверталися до змінних із модуля `math`.

**Модулі** також визначають (define) **функції**. Наприклад, модуль `math` надає ім’я `floor` для функції округлення вниз.  Оскільки ми вже імпортували `math`, ми можемо написати `math.floor(7.5)`, щоб обчислити цілу частину числа 7,5.  (Зверніть увагу, що функція floor повертає найбільше ціле число, що менше або дорівнює заданому числу.)

**Завдання 2.2.** Обчисліть цілу частину числа пі, використовуючи функцію `floor` та змінну `pi` з модуля `math`.  Назвіть результат `floor_of_pi`.


In [ ]:
floor_of_pi = ...
floor_of_pi

In [ ]:
grader.check("q22")

Для довідки нижче наведено ще кілька прикладів функцій із модуля `math`.

Зверніть увагу, що різні функції приймають різну кількість аргументів. Зазвичай у [документації](https://docs.python.org/3/library/math.html) до модуля міститься інформація про те, скільки аргументів потрібно для кожної функції.

*Підказка: якщо натиснути `shift+tab`, перебуваючи поруч із викликом функції, з’явиться документація до цієї функції.*

In [ ]:
# Обчислення логарифмів (логарифм 8 за основою 2).
# Результат 3, тому що 2 у ступені 3 дорівнює 8.
math.log(8, 2)

In [ ]:
# Обчислення квадратних коренів (square roots).
math.sqrt(5)

Існують різні способи імпортування та використання коду із зовнішніх джерел. Метод, який ми використовували вище — `import <назва_модуля>` — імпортує весь модуль і вимагає, щоб для доступу до його коду ми використовували синтаксис `<назва_модуля>.<назва_змінної_або_функції>`.

Ми також можемо імпортувати з модуля тільки конкретну константу або функцію замість усього модуля. Зверніть увагу, що вам не потрібно попередньо використовувати ім’я модуля, щоб посилатися на це конкретне значення. Однак вам слід бути обережними, перепризначаючи імена констант або функцій іншим значенням!

In [ ]:
# Імпорт тільки cos і pi з математики.
# Нам не потрібно використовувати `math.` перед cos або pi
from math import cos, pi
print(cos(pi))

# Однак ми все ще повинні використовувати його перед іншими функціями з модуля `math`
math.log(pi)

Або ми можемо імпортувати всі функції та значення з усього модуля. Тепер нам не потрібно ставити `math` перед будь-якими функціями чи значеннями з модуля `math`.

In [ ]:
# Нарешті, ми можемо імпортувати все з math за допомогою *
# Знову ж таки, в цьому випадку нам не потрібно використовувати `math.` перед функціями з модуля math
from math import *
log(pi)

Іноді, розглянуті способи імпортування окремих (`from math import cos, pi`) або всіх (`from math import *`) функцій бувають не можливими, особливо якщо ім'я функції конфліктує з іншим ім'ям у вашому коді. Декілька різних модулів можуть мати функцію з однаковим ім'ям. Наприклад, функція з ім'ям `log` є в модулі `math`, а також у модулі `numpy`. Хоча обидві функціїї мають однакову назву, вони не є однаковими функціями. Функція `log` з модуля `math` обчислює логарифм числа, тоді як функція `log` з модуля `numpy` обчислює логарифм кожного елемента масиву.

Якщо ви імпортуєте цю функцію з обох модулів, таким чином:
```Python
from math import log
from numpy import log
```
Друга команда імпорту перезаписує першу, і тепер `log` посилається на функцію з модуля `numpy`. Таким чином ви зможете використовувати тільки одну з цих функцій, і ви не зможете використовувати іншу. В наведеному прикладі, якщо ви спробуєте викликати `log(10)`, Python використовуватиме функцію з модуля `numpy`, а не з модуля `math`.
Виходом може бути використання способу імпорта який ми використали першим:
```Python
import math
import numpy
math.log(10)
numpy.log([10,11,12])
```
Тебер все працюватиме вірно.


Якщо ж вам, з тих чи інших причин, не хочеться писати повну назву модуля перед кожною функцію. Є ще один спосіб імпортувати модулі. Ви можете імпортувати модуль і дати йому коротшу назву, наприклад:


In [ ]:
import math as m
import numpy as np
# Тепер ми можемо використовувати скорочену назву `m` замість `math` 
# для доступу до функцій модуля math:
m.log(pi)
# а також скорочену назву `np` замість `numpy` для доступу до функцій модуля numpy:
np.log(pi)

Не переймайтеся надто тим, який тип імпорту використовувати. Зазвичай це питання стилю програмування, яке залишається на розсуд кожного програміста. У цьому курсі ви завжди будете імпортувати необхідні модулі під час виконання підготовчої комірки (наприклад, першої комірки з кодом у цій лабораторній роботі).

Давайте перейдемо до відпрацювання деяких операцій з таблицями, які ви вивчили на лекції!

# 3. Операції з таблицями

Таблиця `farmers_markets.csv` містить дані про фермерські ринки у Сполучених Штатах (дані, пов’язані з [Міністерством сільського господарства США (USDA)](https://www.ams.usda.gov/)). Кожен рядок представляє один такий ринок.

Виконайте наступну комірку, щоб завантажити таблицю `farmers_markets`. Вивід (output) не відбудеться — це очікувано, оскільки комірка містить оператор присвоювання. Оператор присвоювання не створює жодного виводу (він не повертає жодного значення).

In [ ]:
# Просто запустіть цю клітинку

farmers_markets = Table.read_table('farmers_markets.csv')

Давайте розглянемо нашу таблицю, щоб дізнатися, які дані вона містить.

**Завдання 3.1.** Використовуйте метод `show`, щоб відобразити перші 5 рядків таблиці `farmers_markets`.

*Примітка:* Терміни «метод» і «функція» технічно не є синонімами, але в рамках цього курсу ми будемо вживати їх як взаємозамінні.

**Підказка:** `tbl.show(3)` відобразить перші 3 рядки таблиці з назвою `tbl`. Крім того, переконайтеся, що не викликаєте `.show()` без аргументу, оскільки це призведе до аварийного завершення роботи обчислювального ядра (kernel)!

In [ ]:
...

Зверніть увагу, що деякі значення в цій таблиці відсутні, про що свідчить позначка «nan». Це означає, що значення або недоступне (наприклад, якщо ми не знаємо поштової адреси ринку), або не застосовується (наприклад, якщо ринок не має поштової адреси). Ви також помітите, що таблиця містить велику кількість стовпців!

### `num_columns`

Властивість таблиці `num_columns` повертає кількість стовпців у таблиці. («Властивість» — це просто **метод**, який не потрібно викликати, додаючи дужки.)

Приклад виклику: `tbl.num_columns` поверне кількість стовпців у таблиці з іменем `tbl`

**Запитання 3.2.** Використовуйте `num_columns`, щоб визначити кількість стовпців у нашому наборі даних про фермерські ринки.

Присвойте кількість стовпців змінній `num_farmers_markets_columns`.


In [ ]:
num_farmers_markets_columns = ...
print("Таблиця має", num_farmers_markets_columns, "стовпців!")

In [ ]:
grader.check("q32")

### `num_rows`

Подібним чином властивість `num_rows` повідомляє вам, скільки рядків міститься в таблиці.

In [ ]:
# Просто запустіть цю клітинку

num_farmers_markets_rows = farmers_markets.num_rows
print("Таблиця має", num_farmers_markets_rows, "рядків!")

### `select`

Більшість стовпців стосуються конкретних товарів — чи продається на ринку тофу, корм для домашніх тварин тощо. Якщо нас ця інформація не цікавить, вона лише ускладнює сприйняття таблиці. Такі ситуації трапляються частіше, ніж можна було б подумати, адже ті, хто збирає та оприлюднює дані, не завжди заздалегідь знають, як саме користувачі захочуть ними скористатися.

У таких ситуаціях ми можемо скористатися методом таблиці `select`, щоб вибрати лише ті стовпці, які нам потрібні в конкретній таблиці. Цей метод приймає довільну кількість аргументів. Кожен з них має бути назвою стовпця в таблиці. Він повертає нову таблицю, що містить лише ці стовпці. Стовпці розташовуються в тому порядку, *в якому вони були вказані як аргументи*.

Наприклад, значенням `farmers_markets.select(«MarketName», «State»)` є таблиця, що містить лише назву та штат кожного фермерського ринку з таблиці `farmers_markets`.



**Завдання 3.3.** Використовуйте `select`, щоб створити таблицю, яка міститиме лише назву, місто, штат, широту (`y`) та довготу (`x`) кожного ринку.  Назвіть цю нову таблицю `farmers_markets_locations`.

*Підказка:* Будьте уважні при використанні назв стовпців у функції `select`; ретельно перевіряйте великі літери!

In [ ]:
# Запустіть цю клітинку, щоб побачити перший рядок таблиці farmers_markets!
farmers_markets.show(1)

In [ ]:
farmers_markets_locations = ...
farmers_markets_locations

In [ ]:
grader.check("q33")

### `drop`

Команда `drop` виконує ту саму функцію, що й `select`, але видаляє саме ті стовпці, які ви вказали, а не ті, які не вказали. Як і `select`, `drop` повертає нову таблицю.

**Запитання 3.4.** Припустимо, ви просто не хочете, щоб у таблиці `farmers_markets` були стовпці `FMID` та `updateTime`.  Створіть таблицю, яка є копією `farmers_markets`, але не містить цих стовпців.  Назвіть цю таблицю `farmers_markets_without_fmid`.

In [ ]:
farmers_markets_without_fmid = ...
farmers_markets_without_fmid

In [ ]:
grader.check("q34")

Тепер припустімо, що ми хочемо відповісти на кілька запитань про фермерські ринки в США. Наприклад, який(і) ринок(и) має найбільшу довготу (зазначену стовпчиком «x»)? 

Щоб відповісти на це питання, ми відсортуємо `farmers_markets_locations` за довготою.

In [ ]:
farmers_markets_locations.sort('x')

На жаль, це не відповіло на наше запитання, оскільки ми відсортували від найменшої до найбільшої довготи. Щоб переглянути найбільші довготи, нам доведеться відсортувати у зворотному порядку, додавши `descending = True` як другий аргумент.

In [ ]:
farmers_markets_locations.sort('x', descending=True)

(Параметр `descending=True` називається *опціональним аргументом*. Якщо ви не вказали його значення, функція використає його значення за замовчуванням — `False`, тому якщо явно вказати функції `descending=True`, вона виконає сортування у спадному порядку.)

### `sort`

Деякі подробиці щодо сортування:

1. Першим аргументом функції `sort` є назва стовпця, за яким буде здійснюватися сортування.
2. Якщо стовпець містить текст, `sort` сортуватиме за алфавітом; якщо стовпець містить числа, сортування відбуватиметься за числовим значенням — у обох випадках за замовчуванням у порядку зростання. Для літер порядок зростання — від A до Z, а порядок спадання — від Z до A.
3. Значенням виразу `farmers_markets_locations.sort(«x»)` є *копія* таблиці `farmers_markets_locations`; сама таблиця `farmers_markets_locations` не змінюється. Наприклад, якщо ми викликали `farmers_markets_locations.sort(«x»)`, то виконання самого `farmers_markets_locations` все одно поверне несортовану таблицю.
4. Під час сортування таблиці рядки завжди залишаються разом. Немає сенсу сортувати лише один стовпець, а інші залишити без змін. Наприклад, у цьому випадку, якби ми відсортували лише стовпець `x`, усі фермерські ринки мали б неправильні значення довготи.

**Завдання 3.5.** Створіть версію `farmers_markets_locations`, відсортовану за **широтою (`y`)**, з найбільшими значеннями широти на початку.  Назвіть її `farmers_markets_locations_by_latitude`.


In [ ]:
farmers_markets_locations_by_latitude = ...
farmers_markets_locations_by_latitude

In [ ]:
grader.check("q35")

Тепер припустимо, що нам потрібна таблиця всіх фермерських ринків у Каліфорнії. Сортування тут нам не дуже допоможе, оскільки Каліфорнія  знаходиться ближче до середини набору даних.

Натомість ми використовуємо метод таблиці `where`.

In [ ]:
california_farmers_markets = farmers_markets_locations.where('State', are.equal_to('California'))
california_farmers_markets

Поки що не звертайте уваги на синтаксис. Натомість спробуйте прочитати цей рядок так:

> Присвоїти ім’я **`california_farmers_markets`** таблиці, рядки якої відповідають рядкам таблиці **`farmers_markets_locations`**, **`де`** (**`where`**) **`поле «State»`** **`дорівнює` `«California»`**.

### `where`

А тепер давайте трохи детальніше розглянемо це.  `where` приймає 2 аргументи:

1. Назву стовпця.  `where` знаходить рядки, у яких значення цього стовпця відповідають певному критерію.
2. Предикат (умову), що описує критерій, якому повинен відповідати стовпець.

У наведеному вище прикладі предикат викликав функцію `are.equal_to` із потрібним нам значенням — «California».  Незабаром ми розглянемо й інші предикати.

`where` повертає таблицю, яка є копією вихідної таблиці, але **містить лише ті рядки, що відповідають заданому предикату**.

**Завдання 3.6.** Використовуйте `california_farmers_markets`, щоб створити таблицю під назвою `berkeley_markets`, яка містить фермерські ринки в Берклі, Каліфорнія (Berkeley, California.).

In [ ]:
berkeley_markets = ...
berkeley_markets

In [ ]:
grader.check("q36")

Досі ми використовували `where` лише з предикатом, який вимагає, щоб значення у стовпці були *точно* рівними певному значенню. Однак існує багато інших предикатів. Нижче ми навели кілька найпоширеніших, але якщо вам потрібен більш повний перелік, ознайомтеся з розділом про предикати на [довідковій сторінці Python](https://www.data8.org/fa25/reference/#table-filtering-predicates).

|Предикат|Приклад|Результат|
|-|-|-|
|`are.equal_to`|`are.equal_to(50)`|Знайти рядки, значення в яких дорівнюють 50|
|`are.not_equal_to`|`are.not_equal_to(50)`|Знайти рядки, значення в яких не дорівнюють 50|
|`are.above`|`are.above(50)`|Знайти рядки зі значеннями, більшими за (і не рівними) 50|
|`are.above_or_equal_to`|`are.above_or_equal_to(50)`|Знайти рядки зі значеннями, більшими за 50 або рівними 50|
|`are.below`|`are.below(50)`|Знайти рядки зі значеннями, меншими за 50|
|`are.between`|`are.between(2, 10)`|Знайти рядки зі значеннями, більшими або рівними 2 та меншими за 10|
|`are.between_or_equal_to`|`are.between_or_equal_to(2, 10)`|Знайти рядки зі значеннями, більшими або рівними 2 та меншими або рівними 10|

## 4. Аналіз набору даних

Тепер, коли ви ознайомилися з операціями над таблицями, давайте відповімо на цікаве запитання щодо набору даних!

Виконайте код у комірці нижче, щоб завантажити таблицю `imdb`. Вона містить інформацію про 250 фільмів із найвищими рейтингами на IMDb.

In [ ]:
# Просто запустіть цю клітинку

imdb = Table.read_table('imdb.csv')
imdb

Часто нам потрібно виконати кілька операцій — сортування, фільтрування чи інші — щоб перетворити наявну таблицю на щось більш корисне. Ці операції можна виконувати по черзі. Наприклад, припустимо, що у нас є таблиця з назвою `original_tbl`, яка містить стовпці «col1» і «col2»:

```python
first_step = original_tbl.where(“col1”, are.equal_to(12))
second_step = first_step.sort(‘col2’, descending=True)
```

Однак, оскільки значення виразу `original_tbl.where(“col1”, are.equal_to(12))` саме по собі є таблицею, ви можете просто викликати на ній метод таблиці:

```
original_tbl.where(“col1”, are.equal_to(12)).sort(‘col2’, descending=True)
```
Вам слід організувати свою роботу так, як вам зручніше, використовуючи змістовні імена для будь-яких проміжних таблиць, які ви створюєте.

**Завдання 4.1.** Створіть таблицю фільмів, що вийшли в прокат у період з 2010 по 2015 рік (**включно**) та мають оцінку вище 8. Таблиця повинна містити лише стовпці `Title` та `Rating`, **саме в такому порядку**.

Присвойте таблиці ім’я `above_eight`.

*Підказка:* Подумайте про кроки, які потрібно виконати, і спробуйте розташувати їх у логічному порядку. Можете створювати проміжні таблиці для кожного кроку, але не забудьте присвоїти кінцевій таблиці ім’я `above_eight`!

In [ ]:
above_eight = ...
above_eight

In [ ]:
grader.check("q41")

**Завдання 4.2.** Використовуйте `num_rows` (та арифметичні операції), щоб визначити *частку* фільмів у наборі даних, які вийшли в прокат у період 1900–1999 років, та *частку* фільмів у наборі даних, які вийшли в прокат у 2000 році або пізніше.

Присвойте `proportion_in_20th_century` частці фільмів у наборі даних, що вийшли в прокат у 1900–1999 роках, а `proportion_in_21st_century` — частці фільмів у наборі даних, що вийшли в прокат у 2000 році або пізніше.

*Підказка:* *Частка* фільмів, випущених у 1900-х роках, дорівнює *кількості* фільмів, випущених у 1900-х роках, поділеній на *загальну кількість* фільмів.


In [ ]:
num_movies_in_dataset = ...

num_in_20th_century = ...
num_in_21st_century = ...

proportion_in_20th_century = ...
proportion_in_21st_century = ...

print("Пропорція в 20 ст:", proportion_in_20th_century)
print("Пропорція в 21 столітті:", proportion_in_21st_century)

In [ ]:
grader.check("q42")

## 5. Підсумок

Для довідки наводимо таблицю з усіма функціями та методами, які ми розглянули в цій лабораторній роботі. Наступного тижня ми вивчимо ще більше методів, які додамо до цієї таблиці!

|Назва|Приклад|Призначення|
|-|-|-|
|`sort`|`tbl.sort(«N»)`|Створити копію таблиці, відсортовану за значеннями в стовпці|
|`where`|`tbl.where(«N», are.above(2))`|Створення копії таблиці, що містить лише ті рядки, які відповідають певному *предикату*|
|`num_rows`|`tbl.num_rows`|Обчислення кількості рядків у таблиці|
|`num_columns`|`tbl.num_columns`|Обчислити кількість стовпців у таблиці|
|`select`|`tbl.select(«N»)`|Створити копію таблиці, що містить лише деякі стовпці|
|`drop`|`tbl.drop(«N»)`|Створити копію таблиці без деяких стовпців|

**Такер** хоче привітати вас із завершенням лабораторної роботи № 2! Він вважає, що вам варто відсвяткувати це, перекусивши чимось смачненьким.

<img src="tucker.jpeg" alt="A happy golden retriever looking up at the camera" width="300"/>


---

Ви завершили лабораторну роботу!

**Важлива інформація щодо подання роботи:**
- **Виконайте всі тести** і переконайтеся, що всі вони виконані успішно
- **Збережіть** роботу через меню **«Файл»**

**Ви несете відповідальність за те, щоб переконатися, що ваша робота збережена, перш ніж запускати останню комірку.**


---

Щоб ще раз перевірити свою роботу, клітинка нижче повторно запустить усі тести автогрейдера.

In [ ]:
grader.check_all()

## Подання

Перед виконанням наведеної нижче комірки переконайтеся, що ви виконали всі комірки у своєму зошиті послідовно, щоб у результаті відобразилися всі зображення та графіки. Після виконання двох наведених нижче коміркок ваші результати буде передано викладачу.

#### 1) Напишіть своє ім'я та призвище
В наступній комірці замість "Андрій Ковальчук" впишіть свої власні ім'я та призвище (також в лапках).  
Запустіть комірку

In [ ]:
student_name = "Андрій Ковальчук"

#### 2) Наступну комірку просто запустіть

In [3]:
lab_name = "lab02"
comment = ""
r = grader.check_all()
submit_grade(student_name, lab_name, r.total/r.possible*100, comment)